# Spark 4.2 features, live data

Sources: Kafka topics produced by the `generator` container (real AEMO 5-minute dispatch prices from NEMWeb, simulated meters, CDC events, device diagnostics).
Results that matter are also produced continuously by the `stream`, `rtm` and `batch` services into Postgres (`energy` schema).

In [1]:
import time
from pyspark.sql import functions as F
from lab import *

def timed(df):
    df.limit(1000).collect()
    t0 = time.time(); df.collect(); return round(time.time() - t0, 2)

spark = spark_session("nb-spark-4-2", cores="8", spark__sql__shuffle__partitions="2")
spark.version

'4.3.0'

## 1. Change Data Capture: the `CHANGES` clause

The `batch` service keeps `lake.energy.dim_meter` (Iceberg, Auto CDC SCD Type 1) up to date every minute. Each pipeline run commits an Iceberg snapshot,
and its sequence number is the `CHANGES` version.

`CHANGES` needs a catalog implementing Spark 4.2's `TableCatalog.loadChangelog()`. Iceberg has the change data but doesn't implement that hook yet,
so `lab.IcebergChangelogCatalog` (`docker/spark/catalog/`) bridges it. Spark itself does the post-processing: carry-over removal, update detection and net changes.

In [2]:
spark.sql("SELECT latest_sequence_number AS version, timestamp, latest_snapshot_id FROM lake.energy.dim_meter.metadata_log_entries "
          "WHERE latest_sequence_number IS NOT NULL ORDER BY version").show(truncate=False)
latest = spark.sql("SELECT max(latest_sequence_number) FROM lake.energy.dim_meter.metadata_log_entries").first()[0]
start = max(1, latest - 3)
print("latest version:", latest, "| reading changes from version", start)

+-------+-----------------------+-------------------+
|version|timestamp              |latest_snapshot_id |
+-------+-----------------------+-------------------+
|1      |2026-11-06 09:29:02.747|2925000831670043114|
|2      |2026-11-06 09:31:38.468|1939342302338334048|
+-------+-----------------------+-------------------+



latest version: 2 | reading changes from version 1


**Raw** change rows (`deduplicationMode = 'none'`): copy-on-write MERGE rewrites whole files, so unchanged rows show up as identical delete+insert *carry-over* pairs.

In [3]:
raw = spark.sql(f"SELECT * FROM lake.energy.dim_meter CHANGES FROM VERSION {start} WITH ('deduplicationMode' = 'none')")
default = spark.sql(f"SELECT * FROM lake.energy.dim_meter CHANGES FROM VERSION {start}")
print("raw rows:", raw.count(), "| after default carry-over removal:", default.count())
raw.groupBy("_commit_version", "_change_type").count().orderBy("_commit_version", "_change_type").show()

raw rows: 199 | after default carry-over removal: 67


+---------------+------------+-----+
|_commit_version|_change_type|count|
+---------------+------------+-----+
|              1|      insert|   40|
|              2|      delete|   40|
|              2|      insert|   40|
|              3|      delete|   40|
|              3|      insert|   39|
+---------------+------------+-----+



**`computeUpdates`** turns a delete+insert of the same `meter_id` with different content into `update_preimage` / `update_postimage`.

In [4]:
spark.sql(
    f"SELECT _commit_version, _commit_timestamp, _change_type, meter_id, tariff, status, customer "
    f"FROM lake.energy.dim_meter CHANGES FROM VERSION {start} "
    f"WITH ('computeUpdates' = 'true') "
    f"ORDER BY _commit_version, meter_id, _change_type DESC"
).show(20, truncate=False)

+---------------+-----------------------+------------+--------+------+---------+-------------+
|_commit_version|_commit_timestamp      |_change_type|meter_id|tariff|status   |customer     |
+---------------+-----------------------+------------+--------+------+---------+-------------+
|1              |2026-11-06 09:29:02.746|insert      |MTR-0001|flat  |active   |Sam Singh    |
|1              |2026-11-06 09:29:02.746|insert      |MTR-0002|demand|active   |Avery Nguyen |
|1              |2026-11-06 09:29:02.746|insert      |MTR-0003|flat  |active   |Jordan Taylor|
|1              |2026-11-06 09:29:02.746|insert      |MTR-0004|flat  |active   |Riley Smith  |
|1              |2026-11-06 09:29:02.746|insert      |MTR-0005|tou   |active   |Quinn Smith  |
|1              |2026-11-06 09:29:02.746|insert      |MTR-0006|demand|active   |Casey Martin |
|1              |2026-11-06 09:29:02.746|insert      |MTR-0007|flat  |active   |Taylor Smith |
|1              |2026-11-06 09:29:02.746|insert   

**`netChanges`** collapses everything in the range to one net change per meter, the usual input for a downstream incremental load.

In [5]:
spark.sql(
    f"SELECT _change_type, meter_id, tariff, status, customer, _commit_version "
    f"FROM lake.energy.dim_meter CHANGES FROM VERSION {start} TO VERSION {latest} "
    f"WITH ('deduplicationMode' = 'netChanges', 'computeUpdates' = 'true') "
    f"ORDER BY meter_id"
).show(20, truncate=False)

+------------+--------+------+---------+-------------+---------------+
|_change_type|meter_id|tariff|status   |customer     |_commit_version|
+------------+--------+------+---------+-------------+---------------+
|insert      |MTR-0001|flat  |active   |Sam Singh    |1              |
|insert      |MTR-0002|demand|active   |Avery Nguyen |1              |
|insert      |MTR-0003|flat  |active   |Taylor Singh |2              |
|insert      |MTR-0004|demand|active   |Riley Smith  |2              |
|insert      |MTR-0005|tou   |suspended|Quinn Smith  |2              |
|insert      |MTR-0006|demand|active   |Casey Martin |1              |
|insert      |MTR-0007|flat  |active   |Taylor Smith |2              |
|insert      |MTR-0008|tou   |active   |Taylor Wilson|1              |
|insert      |MTR-0009|flat  |suspended|Jordan Kelly |1              |
|insert      |MTR-0010|flat  |active   |Alex Chen    |1              |
|insert      |MTR-0011|tou   |active   |Taylor Kelly |1              |
|inser

Timestamp ranges and the DataFrame API. Timestamp bounds must be literals: an expression such as `current_timestamp() - INTERVAL 1 HOUR` fails in the 4.3 parser.

In [6]:
since = spark.sql("SELECT date_format(current_timestamp() - INTERVAL 10 MINUTES, 'yyyy-MM-dd HH:mm:ss')").first()[0]
spark.sql(f"SELECT _change_type, count(*) FROM lake.energy.dim_meter CHANGES FROM TIMESTAMP '{since}' GROUP BY 1").show()

(spark.read.option("startingVersion", str(start)).option("computeUpdates", "true")
      .changes("lake.energy.dim_meter")
      .groupBy("_change_type").count().show())

+------------+--------+
|_change_type|count(1)|
+------------+--------+
|      insert|      53|
|      delete|      14|
+------------+--------+



+----------------+-----+
|    _change_type|count|
+----------------+-----+
|          insert|   40|
| update_preimage|   13|
|update_postimage|   13|
|          delete|    1|
+----------------+-----+



The `batch` job runs the same query incrementally (`CHANGES FROM VERSION <last loaded> EXCLUSIVE`) and appends to Postgres `dim_meter_change_log`.

In [7]:
jdbc_read(spark, "SELECT change_type, count(*) AS rows, max(commit_version) AS last_version FROM dim_meter_change_log GROUP BY 1").show()
dim = jdbc_read(spark, "dim_meter")

+----------------+----+------------+
|     change_type|rows|last_version|
+----------------+----+------------+
| update_preimage|   7|           2|
|update_postimage|   7|           2|
|          insert|  40|           1|
+----------------+----+------------+



## 2. Vector search in SQL

24-dimension hourly load profile per meter from the 24h interval backfill (`meter_intervals`), then
`vector_normalize`, `vector_cosine_similarity`, `vector_l2_distance`, `vector_avg` and the `NEAREST BY` top-K join.

In [8]:
kafka_batch(spark, "meter_intervals", INTERVAL_SCHEMA).dropDuplicates(["meter_id", "interval_start"]).createOrReplaceTempView("intervals")
dim.select("meter_id", "segment", "region_id").createOrReplaceTempView("meters")

profiles = spark.sql('''
    WITH hourly AS (
        SELECT meter_id, hour(interval_start + INTERVAL 10 HOURS) AS h, sum(kwh) AS kwh
        FROM intervals GROUP BY ALL
    )
    SELECT h.meter_id, m.segment,
           vector_normalize(transform(sort_array(collect_list(struct(h.h, h.kwh))), x -> cast(x.kwh AS float))) AS profile
    FROM hourly h JOIN meters m USING (meter_id)
    GROUP BY h.meter_id, m.segment HAVING count(*) = 24
''').cache()
profiles.createOrReplaceTempView("profiles")
profiles.selectExpr("meter_id", "segment", "vector_norm(profile, 2.0F) AS norm", "slice(profile, 17, 6) AS evening").show(5, truncate=False)

+--------+-----------+----+------------------------------------------------------------------------+
|meter_id|segment    |norm|evening                                                                 |
+--------+-----------+----+------------------------------------------------------------------------+
|MTR-0003|residential|1.0 |[0.1901952, 0.29809418, 0.42529124, 0.4900368, 0.31830895, 0.19768147]  |
|MTR-0008|residential|1.0 |[0.17913961, 0.30082345, 0.41879317, 0.532855, 0.308821, 0.18242101]    |
|MTR-0009|commercial |1.0 |[0.27638614, 0.2936733, 0.08453011, 0.10253492, 0.0794109, 0.08641308]  |
|MTR-0014|residential|1.0 |[0.18650119, 0.30107546, 0.41709384, 0.50754565, 0.31535086, 0.18264583]|
|MTR-0016|residential|1.0 |[0.173124, 0.28966635, 0.43448377, 0.51108766, 0.33249134, 0.1841431]   |
+--------+-----------+----+------------------------------------------------------------------------+
only showing top 5 rows


In [9]:
spark.sql('''
    SELECT segment, count(*) AS meters, vector_avg(profile) AS centroid
    FROM profiles GROUP BY segment
''').createOrReplaceTempView("centroids")

spark.sql('''
    SELECT p.meter_id, p.segment AS actual, c.segment AS nearest_centroid,
           round(vector_cosine_similarity(p.profile, c.centroid), 4) AS cosine,
           round(vector_l2_distance(p.profile, c.centroid), 4) AS l2
    FROM profiles p JOIN centroids c
      EXACT NEAREST 1 BY SIMILARITY vector_cosine_similarity(p.profile, c.centroid)
    ORDER BY p.meter_id
''').show(12)

+--------+-----------+----------------+------+------+
|meter_id|     actual|nearest_centroid|cosine|    l2|
+--------+-----------+----------------+------+------+
|MTR-0001|residential|     residential|0.9993|0.0361|
|MTR-0002|residential|     residential|0.9989|0.0472|
|MTR-0003|residential|     residential|0.9991|0.0418|
|MTR-0004|residential|     residential|0.9994|0.0336|
|MTR-0005| commercial|      commercial|0.9992|0.0391|
|MTR-0006| commercial|      commercial|0.9992|0.0398|
|MTR-0007|residential|     residential|0.9987|  0.05|
|MTR-0008|residential|     residential|0.9995|0.0328|
|MTR-0009| commercial|      commercial|0.9988|0.0497|
|MTR-0010|residential|     residential|0.9992|0.0389|
|MTR-0011| commercial|      commercial|0.9985|0.0548|
|MTR-0012| commercial|      commercial|0.9989|0.0478|
+--------+-----------+----------------+------+------+
only showing top 12 rows


In [10]:
a, b = profiles.alias("a"), profiles.alias("b")
similar = (a.nearestByJoin(b, F.expr("vector_cosine_similarity(a.profile, b.profile)"), 4, "exact", "similarity")
             .where("a.meter_id <> b.meter_id")
             .selectExpr("a.meter_id", "a.segment", "b.meter_id AS similar", "b.segment AS similar_segment",
                         "round(vector_cosine_similarity(a.profile, b.profile), 4) AS cosine"))
similar.orderBy("meter_id", F.desc("cosine")).show(12)

+--------+-----------+--------+---------------+------+
|meter_id|    segment| similar|similar_segment|cosine|
+--------+-----------+--------+---------------+------+
|MTR-0001|residential|MTR-0031|    residential|0.9993|
|MTR-0001|residential|MTR-0010|    residential|0.9992|
|MTR-0001|residential|MTR-0004|    residential|0.9991|
|MTR-0002|residential|MTR-0024|    residential|0.9992|
|MTR-0002|residential|MTR-0010|    residential|0.9991|
|MTR-0002|residential|MTR-0001|    residential|0.9988|
|MTR-0003|residential|MTR-0033|    residential|0.9994|
|MTR-0003|residential|MTR-0014|    residential|0.9993|
|MTR-0003|residential|MTR-0015|    residential| 0.999|
|MTR-0004|residential|MTR-0039|    residential|0.9992|
|MTR-0004|residential|MTR-0025|    residential|0.9992|
|MTR-0004|residential|MTR-0018|    residential|0.9991|
+--------+-----------+--------+---------------+------+
only showing top 12 rows


In [11]:
spark.sql('''
    SELECT a.meter_id, b.meter_id FROM profiles a JOIN profiles b
      APPROX NEAREST 3 BY SIMILARITY vector_cosine_similarity(a.profile, b.profile)
''').explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [meter_id#391, meter_id#1319]
   +- Generate inline(__nearest_matches__#1413), [meter_id#391], false, [meter_id#1319, segment#1325, profile#1333]
      +- Filter ((size(__nearest_matches__#1413, false) > 0) AND isnotnull(__nearest_matches__#1413))
         +- SortAggregate(key=[__qid#1411], functions=[first(meter_id#391, false), max_by(named_struct(meter_id, meter_id#1319, segment, segment#1325, profile, profile#1333), static_invoke(VectorFunctionImplUtils.vectorCosineSimilarity(profile#396, profile#1333, vector_cosine_similarity)), 3, false, 0, 0)])
            +- Sort [__qid#1411 ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(__qid#1411, 2), ENSURE_REQUIREMENTS, [plan_id=2070]
                  +- SortAggregate(key=[__qid#1411], functions=[partial_first(meter_id#391, false), partial_max_by(named_struct(meter_id, meter_id#1319, segment, segment#1325, profile, profile#1333), static_invoke(VectorF

## 3. Native geospatial types

Spark 4.3 OSS ships `GEOMETRY`/`GEOGRAPHY`, WKB I/O (`st_geomfromwkb`, `st_geogfromwkb`, `st_asbinary`), SRIDs (`st_srid`, `st_setsrid`) and Parquet support.
There are no spatial predicates or distance functions yet (`st_distance`, `st_geomfromtext` are absent), so distance below is haversine on lon/lat, ranked with `NEAREST BY`.

In [12]:
import struct

@F.udf("binary")
def point_wkb(lon, lat):
    return struct.pack("<BIdd", 1, 1, lon, lat)

geo = (dim.withColumn("wkb", point_wkb("lon", "lat"))
          .withColumn("geog", F.expr("st_geogfromwkb(wkb)"))
          .withColumn("geom", F.expr("st_setsrid(st_geomfromwkb(wkb), 4326)")))
geo.write.mode("overwrite").parquet("/data/notebooks/meter_geo")
back = spark.read.parquet("/data/notebooks/meter_geo")
back.printSchema()
back.selectExpr("meter_id", "typeof(geog) AS geog_type", "st_srid(geog) AS geog_srid", "typeof(geom) AS geom_type",
                "hex(st_asbinary(geog)) = hex(wkb) AS wkb_roundtrip").show(5, truncate=False)

root
 |-- meter_id: string (nullable = true)
 |-- region_id: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- tariff: string (nullable = true)
 |-- status: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- geog_wkb: binary (nullable = true)
 |-- last_seq: long (nullable = true)
 |-- loaded_at: timestamp (nullable = true)
 |-- wkb: binary (nullable = true)
 |-- geog: geography(4326) (nullable = true)
 |-- geom: geometry(4326) (nullable = true)



+--------+---------------+---------+--------------+-------------+
|meter_id|geog_type      |geog_srid|geom_type     |wkb_roundtrip|
+--------+---------------+---------+--------------+-------------+
|MTR-0001|geography(4326)|4326     |geometry(4326)|true         |
|MTR-0002|geography(4326)|4326     |geometry(4326)|true         |
|MTR-0003|geography(4326)|4326     |geometry(4326)|true         |
|MTR-0004|geography(4326)|4326     |geometry(4326)|true         |
|MTR-0005|geography(4326)|4326     |geometry(4326)|true         |
+--------+---------------+---------+--------------+-------------+
only showing top 5 rows


In [13]:
back.createOrReplaceTempView("m")
jdbc_read(spark, "dim_substation").createOrReplaceTempView("s")
km = '''2 * 6371 * asin(sqrt(pow(sin(radians(s.lat - m.lat) / 2), 2) +
        cos(radians(m.lat)) * cos(radians(s.lat)) * pow(sin(radians(s.lon - m.lon) / 2), 2)))'''
spark.sql(f'''
    SELECT m.meter_id, m.region_id, s.name AS substation, round({km}, 2) AS km
    FROM m JOIN s EXACT NEAREST 1 BY DISTANCE {km}
    ORDER BY km DESC
''').show(10)

+--------+---------+-------------+-----+
|meter_id|region_id|   substation|   km|
+--------+---------+-------------+-----+
|MTR-0002|      SA1| Adelaide CBD|44.88|
|MTR-0017|     NSW1|   Sydney CBD|43.16|
|MTR-0030|     TAS1|    Glenorchy|37.59|
|MTR-0034|     TAS1|       Hobart|37.42|
|MTR-0011|     NSW1|   Sydney CBD|36.74|
|MTR-0012|      SA1| Adelaide CBD|36.23|
|MTR-0006|      SA1| Adelaide CBD|35.54|
|MTR-0033|     TAS1|       Hobart| 35.5|
|MTR-0038|     NSW1|   Sutherland|29.78|
|MTR-0039|     VIC1|Melbourne CBD|29.73|
+--------+---------+-------------+-----+
only showing top 10 rows


## 4. Arrow-optimised Python UDFs by default

No code change: plain `@udf` now plans as `ArrowEvalPython` (eval type 101). Compare with `useArrow=False`.

In [14]:
def to_kwh(wh):
    return wh / 1000.0

arrow_udf = F.udf(to_kwh, "double")
pickled_udf = F.udf(to_kwh, "double", useArrow=False)
df = spark.range(3_000_000).withColumn("wh", F.rand(1) * 5000).cache()
df.count()

for name, u in [("default (Arrow)", arrow_udf), ("useArrow=False", pickled_udf)]:
    q = df.select(F.sum(u("wh")))
    plan = q._jdf.queryExecution().executedPlan().toString()
    dt = timed(q)
    print(f"{name:18s} {dt:6.2f}s  Arrow in plan: {'ArrowEvalPython' in plan}")

default (Arrow)      0.77s  Arrow in plan: True


useArrow=False       1.21s  Arrow in plan: False


## 5. Real-Time Mode from PySpark (stateless)

`trigger(realTime=...)` keeps one long-running batch with records flowing straight through. Sinks: Kafka here.
The `rtm` service runs the same pattern permanently (`jobs/rtm_alerts.py`); latency lands in `fact_alert.latency_ms`.

In [15]:
q = (kafka_stream(spark, "meter_reads", READ_SCHEMA, starting="latest")
     .where("voltage > 245")
     .selectExpr("meter_id AS key",
                 "to_json(named_struct('meter_id', meter_id, 'voltage', voltage, 'source_kafka_ts', kafka_ts)) AS value")
     .writeStream.format("kafka")
     .option("kafka.bootstrap.servers", KAFKA)
     .option("topic", "nb_rtm_out")
     .option("checkpointLocation", f"/data/notebooks/ckpt/rtm_{int(time.time())}")
     .outputMode("update")
     .trigger(realTime="30 seconds")
     .start())
time.sleep(25)
q.stop()

out = kafka_batch(spark, "nb_rtm_out", "meter_id string, voltage double, source_kafka_ts timestamp")
out.selectExpr("(unix_micros(kafka_ts) - unix_micros(source_kafka_ts)) / 1000.0 AS latency_ms") \
   .summary("count", "min", "50%", "90%", "max").show()

+-------+----------+
|summary|latency_ms|
+-------+----------+
|  count|        24|
|    min|  8.000000|
|    50%|      10.0|
|    90%|      18.0|
|    max|218.000000|
+-------+----------+



In [16]:
jdbc_read(spark, "SELECT kind, count(*) AS alerts, percentile_cont(0.5) WITHIN GROUP (ORDER BY latency_ms) AS p50_ms, "
                 "max(latency_ms) AS max_ms FROM fact_alert GROUP BY kind").show()

+------------+------+------+------+
|        kind|alerts|p50_ms|max_ms|
+------------+------+------+------+
|over_voltage|    42|  11.5|  43.0|
| power_spike|    22|  12.5|5824.0|
+------------+------+------+------+



## 6. Governed metrics (metric views)

Define a metric once as YAML, query it with `MEASURE()` from any report.

In [17]:
jdbc_read(spark, "SELECT r.meter_id, r.region_id, m.segment, r.kwh, r.rrp, r.cost_aud FROM fact_meter_read r "
                 "JOIN dim_meter m USING (meter_id) WHERE r.read_ts > now() - interval '1 hour'") \
    .write.mode("overwrite").parquet("/data/notebooks/nb_reads")
spark.sql("CREATE TABLE IF NOT EXISTS nb_reads USING parquet LOCATION '/data/notebooks/nb_reads'")
spark.sql("REFRESH TABLE nb_reads")
spark.sql("DROP VIEW IF EXISTS nb_energy_metrics")
spark.sql('''
CREATE VIEW nb_energy_metrics WITH METRICS LANGUAGE YAML AS $$
version: "0.1"
source: nb_reads
dimensions:
  - name: region_id
    expr: region_id
  - name: segment
    expr: segment
measures:
  - name: kwh
    expr: sum(kwh)
  - name: cost_aud
    expr: sum(cost_aud)
  - name: effective_price_mwh
    expr: sum(cost_aud) / sum(kwh) * 1000
$$''')
spark.sql("SELECT segment, MEASURE(kwh) AS kwh, MEASURE(cost_aud) AS cost, MEASURE(effective_price_mwh) AS aud_per_mwh "
          "FROM nb_energy_metrics GROUP BY segment ORDER BY kwh DESC").show()
spark.sql("SELECT region_id, MEASURE(effective_price_mwh) AS aud_per_mwh FROM nb_energy_metrics GROUP BY region_id").show()

+-----------+------------------+-------------------+------------------+
|    segment|               kwh|               cost|       aud_per_mwh|
+-----------+------------------+-------------------+------------------+
|residential| 5.193182999999989| 0.4918608006572399| 94.71278032321237|
| commercial| 2.717956999999995|0.24930344492326995| 91.72457287708026|
|         ev| 1.108507999999998|0.10431863687167009| 94.10724764428427|
|      solar|0.4826980000000002|0.04830841584000001|100.07999999999997|
+-----------+------------------+-------------------+------------------+



+---------+------------------+
|region_id|       aud_per_mwh|
+---------+------------------+
|     QLD1| 93.20999999999998|
|     TAS1|100.08000000000008|
|     NSW1| 97.30228863181001|
|      SA1| 90.28600712658772|
|     VIC1| 86.18005475886855|
+---------+------------------+

